In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
folder = '/content/drive/MyDrive/Taiwan Food Waste Data'

national = pd.read_csv(f'{folder}/stat_p_150.csv')
national['year_ad'] = national['year'] + 1911
national = national.sort_values('year_ad').reset_index(drop=True)

county = pd.read_csv(f'{folder}/stat_p_45.csv')
county['year_ad'] = county['year'] + 1911

monthly = pd.read_csv(f'{folder}/stat_p_127.csv')
parts = monthly['item1'].str.extract(r'(\d+)年\s*(\d+)?月?')
monthly['year_ad'] = parts[0].astype(int) + 1911
monthly['month'] = pd.to_numeric(parts[1])
monthly['is_annual'] = monthly['month'].isna()

print(national[['year_ad', 'wastegenerated_value', 'wastedisposal_foodwasterecyling_value']])

   year_ad  wastegenerated_value  wastedisposal_foodwasterecyling_value
0     2015               7282863                                 609706
1     2016               7411184                                 575932
2     2017               7851606                                 551332
3     2018               9834046                                 594992
4     2019              10123698                                 498045
5     2020              10247586                                 529567
6     2021              10430497                                 482105
7     2022              11858326                                 488747
8     2023              12148718                                 478565
9     2024              12351875                                 505302


In [ ]:
import plotly.graph_objects as go

# Pick the 2024 row
r = national[national['year_ad'] == 2024].iloc[0]

# The boxes in the chart (numbered 0-9 in this order)
labels = ['Waste generated', 'Incineration', 'Landfill', 'Recyclables',
          'Food waste recovered', 'Other disposal', 'Not properly handled',
          'Compost', 'Pig feed', 'Other food use']

# Each flow goes from a source box to a target box, with a size in tonnes
source = [0, 0, 0, 0, 0, 0, 4, 4, 4]
target = [1, 2, 3, 4, 5, 6, 7, 8, 9]
values = [r['wastedisposal_incineration_value'],
          r['wastedisposal_sanitarylandfill_value'],
          r['wastedisposal_recyclablewaste_value'],
          r['wastedisposal_foodwasterecyling_value'],
          r['wastedisposal_others_value'],
          r['wastenotdisposal_value'],
          r['wastedisposal_foodwasterecyling_composting_value'],
          r['wastedisposal_foodwasterecyling_pigfeed_value'],
          r['wastedisposal_foodwasterecyling_others_value']]

fig = go.Figure(go.Sankey(node=dict(label=labels, pad=20),
                          link=dict(source=source, target=target, value=values)))
fig.update_layout(title="Where Taiwan's household waste went in 2024 (tonnes)")
fig.show()

In 2024, Taiwan generated 12.35 million tonnes of household waste. About 57% was recycled and 33% incinerated, and only about 4% (505,302 tonnes) was recovered as food waste, split mainly between compost and pig feed.

In [ ]:
import plotly.express as px

# Keep the year plus the three food-waste destinations, and give them short names
trend = national[['year_ad',
                  'wastedisposal_foodwasterecyling_composting_value',
                  'wastedisposal_foodwasterecyling_pigfeed_value',
                  'wastedisposal_foodwasterecyling_others_value']].rename(columns={
    'wastedisposal_foodwasterecyling_composting_value': 'Compost',
    'wastedisposal_foodwasterecyling_pigfeed_value': 'Pig feed',
    'wastedisposal_foodwasterecyling_others_value': 'Other use'})

# Reshape from wide (one column per destination) to long (one row per year+destination) for plotting
long = trend.melt(id_vars='year_ad', var_name='Destination', value_name='Tonnes')

fig = px.line(long, x='year_ad', y='Tonnes', color='Destination', markers=True,
              title='Recovered food waste by destination, Taiwan 2015–2024 (tonnes)')
fig.show()
print(trend)

   year_ad  Compost  Pig feed  Other use
0     2015   197107    408524       4076
1     2016   197307    372280       6346
2     2017   204598    343906       2828
3     2018   231676    358229       5087
4     2019   246367    237849      13828
5     2020   261480    243795      24292
6     2021   263829    185619      32657
7     2022   246679    198159      43909
8     2023   232808    197498      48259
9     2024   231369    216361      57572


Pig feed was the main outlet for recovered food waste until 2019, when it fell 42% from its 2015 level. Compost overtook it, but it didn't absorb the lost volume, while other uses grew from almost nothing to about 58,000 tonnes.

In [ ]:
import plotly.express as px

# Recovery rate = food waste recovered ÷ total waste generated, as a percentage
national['recovery_rate_pct'] = (national['wastedisposal_foodwasterecyling_value']
                                 / national['wastegenerated_value'] * 100)

fig = px.bar(national, x='year_ad', y='recovery_rate_pct',
             labels={'year_ad': 'Year', 'recovery_rate_pct': 'Food waste recovered (% of total waste)'},
             title='Share of Taiwan household waste recovered as food waste, 2015–2024')
fig.show()
print(national[['year_ad', 'recovery_rate_pct']].round(2))

   year_ad  recovery_rate_pct
0     2015               8.37
1     2016               7.77
2     2017               7.02
3     2018               6.05
4     2019               4.92
5     2020               5.17
6     2021               4.62
7     2022               4.12
8     2023               3.94
9     2024               4.09


The share of household waste recovered as food waste fell from 8.4% in 2015 to 4.1% in 2024. Part of the early drop may come from a jump in reported total waste in 2018, but the decline continues after 2019 (from 4.9% to 4.1%).

In [ ]:
# Load population and join it to the county waste data by year and county
county = pd.read_csv(f'{folder}/stat_p_45.csv')
county['year_ad'] = county['year'] + 1911
pop = pd.read_csv(f'{folder}/population_by_county.csv')
county_pop = county.merge(pop, on=['year_ad', 'county'])

# Kilograms of food waste recovered per resident
county_pop['food_kg_per_person'] = county_pop['foodwastesrecycled'] * 1000 / county_pop['population']

# Translate names AFTER merging, using the dictionary from Block 6
county_pop['county_en'] = county_pop['county'].replace(english)

c2024 = county_pop[county_pop['year_ad'] == 2024].sort_values('food_kg_per_person', ascending=False)

fig = px.bar(c2024, x='county_en', y='food_kg_per_person',
             labels={'county_en': 'County', 'food_kg_per_person': 'Food waste recovered (kg per person)'},
             title='Food waste recovered per person by county, Taiwan 2024')
fig.show()
print(c2024[['county_en', 'population', 'food_kg_per_person']].round(1))

             county_en  population  food_kg_per_person
34  Lienchiang (Matsu)       13950                57.0
22              Penghu      107901                38.6
23              Tainan     1858651                31.9
41              Kinmen      143601                31.1
29         Chiayi City      262177                28.2
38              Miaoli      532854                26.5
28       Chiayi County      478786                26.3
32          New Taipei     4047001                25.7
36           Kaohsiung     2731412                25.6
24             Taitung      210219                23.4
26            Taichung     2860601                21.8
37             Taoyuan     2338648                19.5
25              Taipei     2490869                18.1
42             Hualien      315374                17.6
43               Yilan      449212                14.9
27            Changhua     1225675                14.7
33              Yunlin      658427                14.0
31        

In [ ]:
# Map Chinese county names to English for the chart
english = {'連江縣': 'Lienchiang (Matsu)', '澎湖縣': 'Penghu', '臺南市': 'Tainan', '金門縣': 'Kinmen',
           '嘉義市': 'Chiayi City', '苗栗縣': 'Miaoli', '嘉義縣': 'Chiayi County', '新北市': 'New Taipei',
           '高雄市': 'Kaohsiung', '臺東縣': 'Taitung', '臺中市': 'Taichung', '桃園市': 'Taoyuan',
           '臺北市': 'Taipei', '花蓮縣': 'Hualien', '宜蘭縣': 'Yilan', '彰化縣': 'Changhua',
           '雲林縣': 'Yunlin', '新竹市': 'Hsinchu City', '新竹縣': 'Hsinchu County', '南投縣': 'Nantou',
           '屏東縣': 'Pingtung', '基隆市': 'Keelung'}
county['county'] = county['county'].replace(english)
pop['county'] = pop['county'].replace(english)

Food waste recovered per resident varied about sixfold across Taiwan's counties in 2024, from 57 kg in Matsu to 9 kg in Keelung. Among main-island counties, Tainan led at 32 kg. Island figures are inflated by visitors who aren't counted as residents.